**Exploratory notebook — historical record.**

The analysis here informed the design of the production pipeline. For production implementations see:
- Data loading → \src/data_loader.py\n- Feature engineering → \src/feature_engineering.py\n
Kept as evidence of the analysis process for academic submission.

# 01 — Exploratory Data Analysis: PaySim Mobile Money Fraud

**Business context:** Before building any model, we need to understand where fraud lives in this dataset, what patterns distinguish it, and what preprocessing decisions (like filtering transaction types) are justified by the data — not by convention.

**Data loading:** Uses `src/data_loader.py` with config-driven stratified sampling.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.feature_engineering import engineer_features
from src.utils import load_config, setup_logging

setup_logging()
config = load_config(PROJECT_ROOT / "config" / "config.yaml")
df = load_data(config)
df.head()

## 1. Class Imbalance

At ~0.1% fraud rate, accuracy is a misleading metric. We report exact counts and visualize the imbalance.

In [ ]:
fraud_count = df["isFraud"].sum()
fraud_rate = df["isFraud"].mean()
print(f"Rows: {len(df):,}")
print(f"Fraud count: {fraud_count:,}")
print(f"Fraud rate: {fraud_rate:.6f} ({fraud_rate*100:.4f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
df["isFraud"].value_counts().plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
ax.set_title("Class Distribution")
ax.set_xticklabels(["Legitimate (0)", "Fraud (1)"], rotation=0)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs/figures/01_class_imbalance.png", dpi=150)
plt.show()

## 2. Fraud Rate by Transaction Type

**Senior insight:** In PaySim, fraud occurs **only** in `TRANSFER`, `CASH_OUT`, and `DEBIT`. `PAYMENT` and `CASH_IN` have zero fraud — these should be filtered **before modeling** to reduce noise and compute cost.

In [ ]:
type_stats = df.groupby("type").agg(
    count=("isFraud", "size"),
    fraud_count=("isFraud", "sum"),
    fraud_rate=("isFraud", "mean"),
).sort_values("fraud_rate", ascending=False)
type_stats

## 3. Balance Inconsistency Analysis

PaySim fraud involves balance manipulation. We test whether `oldbalanceOrg - amount != newbalanceOrig` correlates with the fraud label.

In [ ]:
df["errorBalanceOrig"] = (df["newbalanceOrig"] - (df["oldbalanceOrg"] - df["amount"])).abs()
df["hasBalanceError"] = (df["errorBalanceOrig"] > 0.01).astype(int)

error_fraud_rate = df.groupby("hasBalanceError")["isFraud"].mean()
print("Fraud rate by balance error flag:")
print(error_fraud_rate)
print(f"\nCorrelation: {df['hasBalanceError'].corr(df['isFraud']):.4f}")

## 4. Amount Distribution: Fraud vs Non-Fraud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, color, ax in [(0, "#3498db", axes[0]), (1, "#e74c3c", axes[1])]:
    subset = df.loc[df["isFraud"] == label, "amount"]
    ax.hist(subset, bins=50, color=color, alpha=0.7)
    ax.set_yscale("log")
    ax.set_title(f"Amount — {'Fraud' if label else 'Non-Fraud'}")
    ax.set_xlabel("amount")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs/figures/01_amount_distribution.png", dpi=150)
plt.show()

## 5. Temporal Patterns (step column)

The `step` column simulates hours. We check for temporal clustering of fraud.

In [ ]:
temporal = df.groupby("step")["isFraud"].agg(["sum", "count"])
temporal["fraud_rate"] = temporal["sum"] / temporal["count"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(temporal.index, temporal["fraud_rate"], alpha=0.6)
ax.set_xlabel("step (simulated hour)")
ax.set_ylabel("fraud rate")
ax.set_title("Fraud Rate Over Time")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs/figures/01_temporal_fraud.png", dpi=150)
plt.show()

## 6. Correlation on Engineered Features

Raw feature correlations miss domain signals — we engineer features first, then analyze.

In [ ]:
featured = engineer_features(df, config)
numeric_cols = featured.select_dtypes(include=[np.number]).columns.tolist()
corr = featured[numeric_cols].corr()["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)
print("Top correlations with isFraud:")
print(corr.head(15))